# Genex Live CDC Demo Notebook

Use this notebook **during the meeting**.

## What it does
- loads your real CDC milestone table
- asks **you live** to answer as the parent
- estimates developmental range in all 4 domains
- generates 5 activities per domain
- produces:
  - a 1-row summary table
  - an activity table
  - a JSON export

## Important notes
- CDC only goes to **60 months**, so older children start at the 60-month band.
- If the OpenAI API key is missing or invalid, the notebook falls back to a more varied built-in activity set.

## Run order
1. Run the install cell once if needed.
2. Run the helper-code cell.
3. Run the CDC-load cell.
4. Run the API-key check cell.
5. Run the live demo cell and answer the questions.

In [ ]:
# Run once if needed in this notebook kernel
%pip install pandas openpyxl openai python-dotenv ipython

In [ ]:
# ------------------------------------------------------------
# Imports + setup
# ------------------------------------------------------------
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()

DOMAIN_LABELS = {
    "speech_language": "Speech / Language",
    "motor": "Motor",
    "social_communication": "Social / Communication",
    "adaptive_cognitive": "Adaptive / Cognitive",
}

CATEGORY_TO_DOMAIN = {
    "social and emotial": "social_communication",
    "language and communication": "speech_language",
    "cognitive": "adaptive_cognitive",
    "movement and physical": "motor",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.35,
    "with_help": 0.15,
    "not_yet": 0.0,
    "no": 0.0,
    "not_sure": 0.05,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())

# Better emergency fallback activities so they don't all look the same
FALLBACK_ACTIVITY_LIBRARY = {
    "speech_language": [
        {"title": "Choice board requesting", "materials": "2 preferred snacks or toys, paper or printed icons", "duration_min": 5,
         "instructions": "Offer two choices, name each clearly, and wait for the child to point, gesture, sign, or say the item.",
         "adaptation": "Accept pointing, eye gaze, or single sounds before expecting a word."},
        {"title": "Routine phrase practice", "materials": "daily routine items like cup, shoes, toothbrush", "duration_min": 6,
         "instructions": "Pick one daily routine and repeat a short target phrase such as 'want water' or 'shoes on' throughout the routine.",
         "adaptation": "Use one-word targets first, then build to two-word phrases."},
        {"title": "Picture WH game", "materials": "3-5 simple pictures or a picture book", "duration_min": 7,
         "instructions": "Ask simple who, what, or where questions and model the answer before asking again.",
         "adaptation": "Let the child point to the answer if expressive language is difficult."},
        {"title": "Sound imitation with favorite objects", "materials": "favorite toys that make or suggest sounds", "duration_min": 5,
         "instructions": "Model simple sounds or word approximations during play and pause for imitation.",
         "adaptation": "Reward any attempt at imitation, not just perfect words."},
        {"title": "Mini story retell", "materials": "phone photos from the day or familiar pictures", "duration_min": 8,
         "instructions": "Use one recent event and help the child retell who, what, and where.",
         "adaptation": "Use sentence starters or visual prompts for support."},
    ],
    "motor": [
        {"title": "Cushion obstacle path", "materials": "pillows, towels, or couch cushions", "duration_min": 7,
         "instructions": "Create a short safe path for stepping over, around, and onto soft surfaces.",
         "adaptation": "Add hand support or simplify the path for balance needs."},
        {"title": "Reach-and-place game", "materials": "light toys and a basket", "duration_min": 6,
         "instructions": "Place toys at different heights and encourage reaching, carrying, and placing them in a basket.",
         "adaptation": "Use larger objects if grasping is hard."},
        {"title": "Ball roll and stop", "materials": "soft ball", "duration_min": 5,
         "instructions": "Roll a ball back and forth and add stop/go cues to support movement and control.",
         "adaptation": "Shorten distance or use a larger ball for easier success."},
        {"title": "Sit-to-stand play", "materials": "small chair or firm cushion, favorite toy", "duration_min": 6,
         "instructions": "Encourage repeated sit-to-stand movements to reach a toy or complete a short game.",
         "adaptation": "Allow push-off from hands or provide light trunk support."},
        {"title": "Line walk balance play", "materials": "masking tape on floor", "duration_min": 7,
         "instructions": "Walk along a taped line, pause, and step over small objects placed nearby.",
         "adaptation": "Use a wider line and keep obstacles very low."},
    ],
    "social_communication": [
        {"title": "Turn-taking toy routine", "materials": "favorite toy or game", "duration_min": 6,
         "instructions": "Use clear my-turn/your-turn language and keep turns very short and predictable.",
         "adaptation": "Use a visual turn card or timer if transitions are hard."},
        {"title": "Show-and-share moment", "materials": "interesting object or toy", "duration_min": 5,
         "instructions": "Encourage the child to show an object, look back to the caregiver, and share enjoyment.",
         "adaptation": "Model the action first and praise any eye gaze or shared attention."},
        {"title": "Pretend play routine", "materials": "doll, stuffed animal, toy food, or household items", "duration_min": 8,
         "instructions": "Model one pretend action at a time, such as feeding a doll or tucking it into bed.",
         "adaptation": "Start with real objects and copy the child's actions."},
        {"title": "Name response game", "materials": "small reinforcers like bubbles or stickers", "duration_min": 5,
         "instructions": "Call the child's name during a playful activity and reward quick orientation or response.",
         "adaptation": "Reduce distractions and use an excited tone or movement cue."},
        {"title": "Simple rule game", "materials": "ball, puzzle pieces, or simple board game parts", "duration_min": 7,
         "instructions": "Practice one easy rule such as wait, roll, or give with lots of repetition.",
         "adaptation": "Use one-step rules only and preview them visually if needed."},
    ],
    "adaptive_cognitive": [
        {"title": "Two-step helper job", "materials": "laundry basket, socks, cups, or other household items", "duration_min": 6,
         "instructions": "Give a short helper routine like get socks then put in basket.",
         "adaptation": "Use gestures or pictures for each step."},
        {"title": "Visual cleanup routine", "materials": "toy bins and simple picture labels", "duration_min": 7,
         "instructions": "Practice cleanup with a short visual sequence and immediate praise after each step.",
         "adaptation": "Use only 2-3 items to clean up at first."},
        {"title": "Matching and sorting", "materials": "blocks, utensils, socks, or colored cards", "duration_min": 6,
         "instructions": "Sort familiar objects by type, color, or function.",
         "adaptation": "Use very distinct categories first, then increase difficulty."},
        {"title": "Mini task board", "materials": "paper or whiteboard with 2-3 tasks", "duration_min": 8,
         "instructions": "Make a tiny board for snack prep, dressing, or toy cleanup and check off each step.",
         "adaptation": "Keep steps concrete and physically visible."},
        {"title": "Cause-and-effect problem play", "materials": "simple puzzle or pop-up toy", "duration_min": 5,
         "instructions": "Use a toy or household task that rewards trial and error, and model one solution at a time.",
         "adaptation": "Provide hand-over-hand support only if truly needed, then fade it."},
    ],
}

def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put the file in the repo root or update the notebook path."
    )

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    cdc_df = pd.read_excel(path)
    cdc_df.columns = [str(c).strip().lower() for c in cdc_df.columns]
    cdc_df = cdc_df.rename(columns={"category ": "category", "milestone ": "milestone"})
    cdc_df["category"] = cdc_df["category"].astype(str).str.strip().str.lower()
    cdc_df["milestone"] = cdc_df["milestone"].astype(str).str.strip()
    cdc_df["domain"] = cdc_df["category"].map(CATEGORY_TO_DOMAIN)
    cdc_df = cdc_df.dropna(subset=["domain"]).copy()
    cdc_df["months"] = cdc_df["months"].astype(int)
    cdc_df["question_id"] = [
        f"{row.domain}_{int(row.months)}_{i}" for i, row in enumerate(cdc_df.itertuples(), start=1)
    ]
    return cdc_df, path

def get_age_bands(cdc_df):
    return sorted(int(x) for x in cdc_df["months"].dropna().unique())

def nearest_cdc_band(cdc_df, months):
    bands = get_age_bands(cdc_df)
    capped = min(int(months), max(bands))
    eligible = [m for m in bands if m <= capped]
    return eligible[-1] if eligible else bands[0]

def get_band_questions(cdc_df, domain, month, max_q=2):
    qdf = cdc_df[(cdc_df["domain"] == domain) & (cdc_df["months"] == int(month))].copy()
    return qdf.head(max_q).to_dict("records")

def normalize_answer(text):
    value = str(text).strip().lower().replace(" ", "_")
    if value in VALID_ANSWERS:
        return value
    return None

def prompt_for_answer(question_text):
    while True:
        ans = input(f"{question_text}\nAnswer (yes / sometimes / with_help / no / not_sure): ").strip().lower()
        normalized = normalize_answer(ans)
        if normalized is not None:
            return normalized
        print("Please enter one of: yes, sometimes, with_help, no, not_sure")

def run_domain_interview_live(cdc_df, domain, chronological_months, max_steps=5, max_q_per_step=2):
    bands = get_age_bands(cdc_df)
    current_idx = bands.index(nearest_cdc_band(cdc_df, chronological_months))
    asked = []
    visited = []

    print("\n" + "=" * 80)
    print(f"Starting {DOMAIN_LABELS[domain]} interview. CDC anchor: {bands[current_idx]} months")
    print("=" * 80)

    for step in range(max_steps):
        if current_idx < 0 or current_idx >= len(bands):
            break
        band = bands[current_idx]
        if band in visited:
            break
        visited.append(band)

        questions = get_band_questions(cdc_df, domain, band, max_q=max_q_per_step + confirmation_q_per_step)
        if not questions:
            break

        scores = []
        print(f"\n--- {DOMAIN_LABELS[domain]} | asking {band}-month milestone(s) ---")
        for q in questions:
            answer = prompt_for_answer(q["milestone"])
            score = ANSWER_SCORES[answer]
            scores.append(score)
            asked.append({
                "band_month": band,
                "question_id": q["question_id"],
                "question": q["milestone"],
                "answer": answer,
                "score": score,
            })

        avg_score = sum(scores) / len(scores) if scores else 0

        if avg_score >= 0.9:
            print(f"Strong YES at {band} months → try one band higher.")
            if current_idx == len(bands) - 1:
                break
            current_idx += 1
        elif avg_score < 0.25:
            print(f"Mostly NO at {band} months → go one band lower.")
            if current_idx == 0:
                break
            current_idx -= 1
        else:
            print(f"Mixed / partial at {band} months → stop here and use a more conservative estimate.")
            break

    band_scores = {}
    for item in asked:
        band_scores.setdefault(item["band_month"], []).append(item["score"])
    band_scores = {month: round(sum(vals) / len(vals), 2) for month, vals in band_scores.items()}

    passed = [m for m, score in sorted(band_scores.items()) if score >= 0.9]
    partial = [m for m, score in sorted(band_scores.items()) if 0.25 <= score < 0.9]
    failed = [m for m, score in sorted(band_scores.items()) if score < 0.25]

    bands = get_age_bands(cdc_df)

    # Conservative estimate: use the last strong pass, not the next partial band
    if passed:
        anchor = max(passed)
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = anchor
    elif partial:
        anchor = min(partial)
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = anchor
    else:
        anchor = min(failed) if failed else bands[0]
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = lower

    return {
        "estimated_range_months": f"{lower}-{upper}",
        "estimated_anchor_month": int(lower),
        "confidence": round(min(0.95, 0.58 + 0.06 * len(asked)), 2),
        "asked": asked,
        "band_scores": band_scores,
    }
def run_domain_interview_live(cdc_df, domain, chronological_months, max_steps=5, max_q_per_step=2, confirmation_q_per_step=2):
    bands = get_age_bands(cdc_df)
    current_idx = bands.index(nearest_cdc_band(cdc_df, chronological_months))
    asked = []
    visited = []

    print("\\n" + "=" * 80)
    print(f"Starting {DOMAIN_LABELS[domain]} interview. CDC anchor: {bands[current_idx]} months")
    print("=" * 80)

    for step in range(max_steps):
        if current_idx < 0 or current_idx >= len(bands):
            break

        band = bands[current_idx]
        if band in visited:
            break
        visited.append(band)

        # Ask the normal questions plus 2 extra confirmation questions if available
        questions = get_band_questions(
            cdc_df,
            domain,
            band,
            max_q=max_q_per_step + confirmation_q_per_step
        )

        if not questions:
            break

        scores = []
        print(f"\\n--- {DOMAIN_LABELS[domain]} | asking {band}-month milestone(s) ---")

        for q in questions:
            answer = prompt_for_answer(q["milestone"])
            score = ANSWER_SCORES[answer]
            scores.append(score)
            asked.append({
                "band_month": band,
                "question_id": q["question_id"],
                "question": q["milestone"],
                "answer": answer,
                "score": score,
            })

        avg_score = sum(scores) / len(scores) if scores else 0

        if avg_score >= 0.9:
            print(f"Strong YES at {band} months → try one band higher.")
            if current_idx == len(bands) - 1:
                break
            current_idx += 1

        elif avg_score < 0.25:
            print(f"Mostly NO at {band} months → go one band lower.")
            if current_idx == 0:
                break
            current_idx -= 1

        else:
            print(f"Mixed / partial at {band} months → stop here and use a more conservative estimate.")
            break

    band_scores = {}
    for item in asked:
        band_scores.setdefault(item["band_month"], []).append(item["score"])
    band_scores = {month: round(sum(vals) / len(vals), 2) for month, vals in band_scores.items()}

    passed = [m for m, score in sorted(band_scores.items()) if score >= 0.9]
    partial = [m for m, score in sorted(band_scores.items()) if 0.25 <= score < 0.9]
    failed = [m for m, score in sorted(band_scores.items()) if score < 0.25]

    bands = get_age_bands(cdc_df)

    # Conservative estimate: use the last strong pass, not the next partial band
    if passed:
        anchor = max(passed)
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = anchor
    elif partial:
        anchor = min(partial)
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = anchor
    else:
        anchor = min(failed) if failed else bands[0]
        lower = bands[max(bands.index(anchor) - 1, 0)]
        upper = lower

    return {
        "estimated_range_months": f"{lower}-{upper}",
        "estimated_anchor_month": int(lower),
        "confidence": round(min(0.95, 0.55 + 0.05 * len(asked)), 2),
        "asked": asked,
        "band_scores": band_scores,
    }
def fallback_activities_for_domain(domain, profile, child):
    items = []
    for idx, item in enumerate(FALLBACK_ACTIVITY_LIBRARY[domain], start=1):
        items.append({
            "activity_number": idx,
            "title": item["title"],
            "why_this_fits": (
                f"Fits a child who is {child['chronological_age_months']} months old with "
                f"{DOMAIN_LABELS[domain]} estimated around {profile['estimated_range_months']} months "
                f"and condition '{child['condition']}'."
            ),
            "instructions": item["instructions"],
            "materials": item["materials"],
            "duration_min": item["duration_min"],
            "adaptation": item["adaptation"],
        })
    return items

def generate_activities_with_llm(child, domain_results, model="gpt-4o-mini", use_llm=True):
    if (not use_llm) or (OpenAI is None) or (not os.environ.get("OPENAI_API_KEY")):
        return {
            domain: fallback_activities_for_domain(domain, result, child)
            for domain, result in domain_results.items()
        }

    client = OpenAI()

    payload = {
        "child_name": child["name"],
        "chronological_age_months": child["chronological_age_months"],
        "condition": child["condition"],
        "caregiver_concern": child["concern_text"],
        "daily_time_min": child["daily_time_min"],
        "estimated_development": {
            domain: {
                "estimated_range_months": result["estimated_range_months"],
                "confidence": result["confidence"],
                "evidence": [
                    {"band_month": item["band_month"], "question": item["question"], "answer": item["answer"]}
                    for item in result["asked"][:4]
                ],
            }
            for domain, result in domain_results.items()
        },
    }

    system_prompt = '''
You are helping generate home activities for Genex.
Return exactly 5 practical, respectful home activities for each domain:
speech_language, motor, social_communication, adaptive_cognitive.

Rules:
- Be supportive and non-diagnostic.
- Activities must fit the child's chronological age, developmental estimate, and condition.
- They must not sound babyish for an older child with developmental delay.
- Keep them realistic for home and short enough for caregivers.
- Avoid medical claims or therapy promises.
- Include varied materials and varied adaptations across activities.
- Return strict JSON only.

Required JSON format:
{
  "speech_language": [{"activity_number":1,"title":"...","why_this_fits":"...","instructions":"...","materials":"...","duration_min":5,"adaptation":"..."}],
  "motor": [...],
  "social_communication": [...],
  "adaptive_cognitive": [...]
}
'''

    try:
        response = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(payload)},
            ],
        )
        content = response.choices[0].message.content
        parsed = json.loads(content)

        for domain in DOMAIN_LABELS:
            if domain not in parsed or not isinstance(parsed[domain], list) or len(parsed[domain]) == 0:
                parsed[domain] = fallback_activities_for_domain(domain, domain_results[domain], child)

        return parsed
    except Exception as e:
        print(f"LLM activity generation failed: {e}")
        return {
            domain: fallback_activities_for_domain(domain, result, child)
            for domain, result in domain_results.items()
        }

def build_summary_row(child, domain_results):
    return {
        "name": child["name"],
        "condition": child["condition"],
        "chronological_age_months": child["chronological_age_months"],
        "speech_range": domain_results["speech_language"]["estimated_range_months"],
        "motor_range": domain_results["motor"]["estimated_range_months"],
        "social_range": domain_results["social_communication"]["estimated_range_months"],
        "adaptive_range": domain_results["adaptive_cognitive"]["estimated_range_months"],
    }

def build_activity_table(child, activities_by_domain):
    rows = []
    for domain, items in activities_by_domain.items():
        for item in items:
            rows.append({
                "name": child["name"],
                "condition": child["condition"],
                "domain": domain,
                "activity_number": item.get("activity_number"),
                "title": item.get("title"),
                "duration_min": item.get("duration_min"),
                "materials": item.get("materials"),
                "why_this_fits": item.get("why_this_fits"),
                "instructions": item.get("instructions"),
                "adaptation": item.get("adaptation"),
            })
    return pd.DataFrame(rows)

def run_live_demo(cdc_df, use_llm=True):
    print("Enter the child profile.")
    name = input("Child name: ").strip() or "Demo Child"
    chronological_age_months = int(input("Chronological age in months (e.g. 36, 60, 72): ").strip())
    condition = input("Condition / diagnosis (or 'none'): ").strip() or "none"
    concern_text = input("Main parent concern: ").strip() or "General developmental support"
    daily_time_min = int(input("How many minutes per day does the family have? ").strip() or "10")

    child = {
        "name": name,
        "chronological_age_months": chronological_age_months,
        "condition": condition,
        "concern_text": concern_text,
        "daily_time_min": daily_time_min,
    }

    domain_results = {}
    for domain in DOMAIN_LABELS:
        domain_results[domain] = run_domain_interview_live(cdc_df, domain, chronological_age_months)

    activities_by_domain = generate_activities_with_llm(child, domain_results, use_llm=use_llm)

    result = {
        "child": child,
        "developmental_profile": domain_results,
        "activities_by_domain": activities_by_domain,
        "note": "CDC milestones only go to 60 months, so older children start from the 60-month band.",
    }

    summary_df = pd.DataFrame([build_summary_row(child, domain_results)])
    activity_df = build_activity_table(child, activities_by_domain)

    return result, summary_df, activity_df

In [ ]:
cdc_df, cdc_path = load_cdc_table()
print("Loaded CDC file from:", cdc_path)
print("Age bands:", get_age_bands(cdc_df))
cdc_df.head()

In [ ]:
print("OPENAI_API_KEY visible:", bool(os.environ.get("OPENAI_API_KEY")))
if os.environ.get("OPENAI_API_KEY"):
    print("Key is present in notebook environment.")
else:
    print("Key is missing. Fallback activities will be used.")

In [ ]:
# Set to False if the API key is not working and you want to force fallback activities.
USE_LLM = True

live_result, live_summary_df, live_activity_df = run_live_demo(cdc_df, use_llm=USE_LLM)

In [ ]:
live_summary_df

In [ ]:
live_activity_df.head(20)

In [ ]:
# Export the live demo outputs
from pathlib import Path
import json

Path("live_demo_output.json").write_text(json.dumps(live_result, indent=2), encoding="utf-8")
live_summary_df.to_csv("live_demo_summary.csv", index=False)
live_activity_df.to_csv("live_demo_activities.csv", index=False)

print("Saved:")
print(Path("live_demo_output.json").resolve())
print(Path("live_demo_summary.csv").resolve())
print(Path("live_demo_activities.csv").resolve())